# Adding Pipelines

A pipeline connects functions together and defines when they run. You can configure a pipeline to trigger automatically when a file with certain tags is uploaded, or to run on demand against any file you choose. This tutorial shows you how to build a pipeline from uploaded functions, upload it to DataLab, and trigger it manually.

## Setup

In [ ]:
from gfhub import nodes
import gfhub

client = gfhub.Client()

## Prerequisites

This tutorial builds directly on tutorial 3. The `csv_to_parquet_example` function must already be uploaded to your organization. You can confirm it is available by listing your functions.

In [ ]:
functions = client.list_functions()
function_names = [f["name"] for f in functions]
print("Available functions:", function_names)

assert "csv_to_parquet_example" in function_names, (
    "csv_to_parquet_example not found. Run tutorial 3 first to upload it."
)

## Build the pipeline

A pipeline is a graph of nodes connected by edges. You define it using `gfhub.Pipeline` and the helper functions in `gfhub.nodes`.

The available node types are:
- `nodes.on_file_upload(tags=[...])`: triggers the pipeline automatically when a file with matching tags is uploaded
- `nodes.on_manual_trigger()`: allows triggering the pipeline manually against any file
- `nodes.load()`: loads the input file from storage and passes it downstream
- `nodes.function(function="name")`: runs an uploaded function
- `nodes.save()`: saves the output files back to DataLab

You connect nodes using the `>>` operator and add edges to the pipeline with `+=`.

In [ ]:
p = gfhub.Pipeline()

# Trigger nodes: run automatically on .csv uploads, or manually on demand
p.auto_trigger = nodes.on_file_upload(tags=[".csv"])
p.manual_trigger = nodes.on_manual_trigger()

# Load the input file
p.load = nodes.load()

# Processing step: convert CSV to Parquet using the function from tutorial 3
p.to_parquet = nodes.function(function="csv_to_parquet_example")

# Save the output with a tag so it is easy to find later
p.save = nodes.save(tags=["processed"])

# Connect the graph
p += p.auto_trigger >> p.load
p += p.manual_trigger >> p.load
p += p.load >> p.to_parquet
p += p.to_parquet >> p.save

print(p)

## Upload the pipeline

Upload the pipeline by giving it a name. If a pipeline with the same name already exists it will be updated. The returned object contains the pipeline `id` which you can use to enable, disable, or trigger it.

In [ ]:
import json

result = client.add_pipeline("csv_to_parquet_pipeline_example", p)
print(json.dumps(result, indent=2, default=str))

## Trigger the pipeline manually

Once the pipeline is uploaded you can run it against any file using `trigger_pipeline`. You pass the pipeline name and one or more file `id` values (from `query_files` or `add_file`). The call returns a job object that you can poll to track progress.

In [ ]:
# Find a CSV file to process
csv_files = client.query_files(tags=[".csv"])

if csv_files:
    file_id = csv_files[0]["id"]
    job = client.trigger_pipeline("csv_to_parquet_pipeline_example", file_id)
    print(f"Job IDs: {job['job_ids']}")

    # trigger_pipeline returns a list of job IDs, one per file
    final = client.wait_for_job(job["job_ids"][0])
    print(f"Status: {final['status']}")
else:
    print("No CSV files found. Upload one first using the upload tutorial.")